In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tqdm
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_toolkits.utils.data_io import most_common_image_ext
from fundus_vessels_toolkit import VTree
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import AVSegToTree, GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TreeTopology, optimal_lines
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees
from fundus_vessels_toolkit.vparameters import parametrize_bifurcations, parametrize_branches

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [ ]:
BL = Path("/home/gaby/Lab/DATA/Fundus/CLSA/BL/")
RAW = "1-images"
AV_FVT = "2-av-pred_FVT"

yy, xx = np.meshgrid(np.arange(1024), np.arange(1024), indexing="ij")
yy -= 512
xx -= 512
roi = np.linalg.norm(np.stack([xx, yy], axis=-1), axis=-1) < 512

IMGS = [_.stem for _ in (BL / RAW).glob(f"*.jpeg")]
IMGS = [stem for stem in IMGS if not (BL / AV_FVT / (stem + ".png")).exists()]

for img in tqdm.tqdm(IMGS):
    fundus = FundusData(image=BL / RAW / (img + ".jpeg"), roi_mask=roi)
    segment_av(fundus)
    fundus.write_image(av=BL / AV_FVT)

 28%|██▊       | 4363/15723 [03:25<08:51, 21.39it/s]

In [2]:
BL = Path("/home/gaby/Lab/DATA/Fundus/CLSA/BL/")
F1 = Path("/home/gaby/Lab/DATA/Fundus/CLSA/F1/")
RAW = "1-images"
AV_Auto = "2-av-pred_Automorph"
AV_FVT = "2-av-pred_FVT"
AV_VASCX = "2-av-pred_VascX"
OD = "2-od"
MACULA = "2-mac"

av2tree_naive = NaiveAVSegToTree()
av2tree_heur = AVSegToTree()
av2tree_gnn = GNNAVSegToTree()

IMGS = sorted({_.stem for _ in (BL / RAW).glob(f"*.jpeg")} & {_.stem for _ in (F1 / RAW).glob(f"*.jpeg")})
len(IMGS)

41153

In [ ]:
IMG = "52335867_retinal_right"  # IMGS[61]
seg_model = "fvt"


def load_fundus(img, path, seg_model="fvt"):
    img_png = img + ".png"
    av = {"fvt": AV_FVT, "automorph": AV_Auto, "vascx": AV_VASCX}[seg_model]
    fundus = FundusData(image=path / RAW / (img + ".jpeg"), od=path / OD / img_png, macula=path / MACULA / img_png)
    if (path / av / img_png).exists():
        fundus.update(av=path / av / img_png, inplace=True)
    else:
        segment_av(fundus)
        fundus.write_image(av=path / av / img_png, on_exists="skip")
    return fundus.remove_od_from_vessels()


fundus_bl = load_fundus(IMG, BL, seg_model=seg_model)
fundus_f1 = load_fundus(IMG, F1, seg_model=seg_model)
m = Mosaic((2, 3), cols_titles=["input", "GNN", "Heuristique"], rows_titles=["BL", "F1"], cell_height=500)

fundus_bl.draw(view=m[0, 0])
draw_trees(av2tree_naive(fundus_bl), view=m[0, 0], bspline_dir=True, interactive=True)
m[0, 1].add_image(fundus_bl.image)
draw_trees(av2tree_gnn(fundus_bl), view=m[0, 1], bspline_dir=True, interactive=True)
m[0, 2].add_image(fundus_bl.image)
draw_trees(av2tree_heur(fundus_bl), view=m[0, 2], bspline_dir=True, interactive=True)


fundus_f1.draw(view=m[1, 0])
draw_trees(av2tree_naive(fundus_f1), view=m[1, 0], bspline_dir=True, interactive=True)
m[1, 1].add_image(fundus_f1.image)
draw_trees(av2tree_gnn(fundus_f1), view=m[1, 1], bspline_dir=True, interactive=True)
m[1, 2].add_image(fundus_f1.image)
draw_trees(av2tree_heur(fundus_f1), view=m[1, 2], bspline_dir=True, interactive=True)

m

/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vtree.py:414: UserWarning: The geometric data contains duplicated nodes coordinates.
  super().__init__(
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vtree.py:414: UserWarning: The geometric data contains duplicated nodes coordinates.
  super().__init__(
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vtree.py:414: UserWarning: The geometric data contains duplicated nodes coordinates.
  super().__init__(
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vtree.py:414: UserWarning: The geometric data contains duplicated nodes coordinates.
  super().__init__(


GridBox(children=(HTML(value='<span/>'), HTML(value='<h3 style="text-align: center;">input</h3>'), HTML(value=…

In [ ]:
from fundus_vessels_toolkit.vmatching.registration import register_graphs


src_trees = av2tree_gnn(fundus_bl)

register_graphs()

In [ ]:
from fundus_vessels_toolkit.pipelines import avseg_to_tree


stats = []
for img in IMGS[:2]:
    stat = {"img": img}
    stat |= {
        (stage, method, seg, art): parametrize_bifurcations(tree).mean()
        for stage, path in {"bl": BL, "f1": F1}.items()
        for seg, fundus in {seg: load_fundus(img, path, seg_model=seg) for seg in ["fvt", "automorph", "vascx"]}.items()
        for method, trees in {
            "naive": av2tree_naive(fundus),
            "gnn": av2tree_gnn(fundus),
            "heur": av2tree_heur(fundus),
        }.items()
        for art, tree in {"art": trees[0], "vei": trees[1]}.items()
    }
    stats.append(stat)